In [ ]:
import os
from openai import OpenAI

anthropic_client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/"
)

In [ ]:
messages = [
    {"role": "user", "content": "hello"}
]

reponse = anthropic_client.chat.completions.create(
    model="claude-sonnet-4-5",
    messages=messages
)

print(reponse.choices[0].message.content)

In [ ]:
import rag
index = rag.initialize_index()

In [ ]:
class ChatCompletionsRAG(rag.RAG):

    def llm(self, user_prompt):
        messages = [
            {"role": "system", "content": self.rag_instructions},
            {"role": "user", "content": user_prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model_name,
            messages=messages,
        )

        return response.choices[0].message.content

In [ ]:
response_rag = ChatCompletionsRAG(
    index,
    anthropic_client,
    model_name="claude-sonnet-4-5"
)

response = response_rag.rag("llm as a judge")
print(response[:50] + "...")

# Native Anthropic SDK

In [ ]:
from anthropic import Anthropic
anthropic_client = Anthropic()

In [ ]:
messages = [
    {"role": "user", "content": "hello"}
]

message = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=messages
)

print(message.model_dump_json(indent=2))
print(message.content[0].text)

In [ ]:
instructions = "You are a helpful assistant. Reply with emojis."

messages = [
    {"role": "user", "content": "Hello"}
]

message = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=instructions,
    messages=messages
)

print(message.content[0].text)

In [ ]:
instructions = """
You're an assistant that can make jokes. Always find out the name of
the person to make the jokes personalized. Once you know the name,
make the joke about them.
""".strip()

messages = [
    {"role": "user", "content": "tell me a joke"}
]

message = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=instructions,
    messages=messages
)

print(message.content[0].text)

In [ ]:
messages.append({"role": "assistant", "content": message.content[0].text})
messages.append({"role": "user", "content": "Alexey"})

In [ ]:
message = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=instructions,
    messages=messages
)

print(message.content[0].text)

### Streaming

In [ ]:
messages = [
    {"role": "user", "content": "Tell me a bedtime story (100 words)"}
]

with anthropic_client.messages.stream(
    max_tokens=1024,
    messages=messages,
    model="claude-sonnet-4-5",
) as stream:
  for text in stream.text_stream:
      print(text, end="", flush=True)

### Structured Output

In [ ]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [ ]:
instructions = "Extract the event information."
content = "Alice and Bob are going to a science fair on Friday."

messages = [{"role": "user", "content": content}]

response = anthropic_client.beta.messages.parse(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    betas=["structured-outputs-2025-11-13"],
    messages=messages,
    output_format=CalendarEvent,
)

print(response.parsed_output)

### Structured Output with Tools

In [ ]:
structured_output_tool = {
    "name": "final_result",
    "description": "Tool for structured output",
    "input_schema": CalendarEvent.model_json_schema()
}

In [ ]:
response = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=messages,
    tools=[structured_output_tool],
    tool_choice={"type": "tool", "name": structured_output_tool['name']}
)

In [ ]:
event = CalendarEvent.model_validate(response.content[0].input)
print(event)

### RAG with Anthropic

In [ ]:
class AnthropicRAG(rag.RAG):

    def llm(self, user_prompt):
        messages = [{"role": "user", "content": user_prompt}]

        response = self.llm_client.beta.messages.parse(
            model=self.model_name,
            max_tokens=1024,
            betas=["structured-outputs-2025-11-13"],
            system=self.rag_instructions,
            messages=messages,
            output_format=self.output_type,
        )

        return response.parsed_output

In [ ]:
anthropic_rag = AnthropicRAG(
    index,
    anthropic_client,
    model_name='claude-sonnet-4-5'
)

response = anthropic_rag.rag("llm as a judge")
print(response.answer[:50] + "...")